In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- bench_interp_time_regular ---

# --- bench_interp_time_rmse ---

print("✅ Fixtures loaded")

✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_bench_interp_time_regular(df=None):
    if df is None:
        df = pd.DataFrame({"date":["2020-01-01"],"value":[1.0],"station_id":["s1"]})
    df = df.dropna()
    station_ids = df.station_id.tolist()
    first_station_id = station_ids[0]
    return df[df["station_id"] == first_station_id]
    return first_station_id

def before_bench_interp_time_rmse():
    def get_rmse(regular_values: pd.Series, interpolated_values: pd.Series):
        diff = (regular_values.reset_index(drop=True) - interpolated_values.reset_index(drop=True)).dropna()
        n = diff.size
        return ((diff**2).sum() / n) ** 0.5
    return get_rmse

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_bench_interp_time_regular(df=None):
    if df is None:
        df = pl.DataFrame({"date":pl.Series(["2020-01-01"],dtype=pl.Utf8),"value":[1.0],"station_id":["s1"]})
    df = df.drop_nulls()
    station_ids = df.get_column("station_id").to_list()
    first_station_id = station_ids[0]
    return df.filter(pl.col("station_id") == first_station_id)

def gen_bench_interp_time_rmse():

    def get_rmse(regular_values: pl.Series, interpolated_values: pl.Series):
        diff = (regular_values - interpolated_values).drop_nulls()
        n = diff.len()
        return ((diff ** 2).sum() / n) ** 0.5
    return get_rmse

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: bench_interp_time_regular ===

def _regular_pair(frame_pd):
    return frame_pd, pl.from_pandas(frame_pd)

# L1 smoke – generated
try:
    _r = gen_bench_interp_time_regular()
    print("✅ L1 smoke gen_bench_interp_time_regular: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_bench_interp_time_regular: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_bench_interp_time_regular()
    print("✅ L1 smoke before_bench_interp_time_regular: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_bench_interp_time_regular: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_bench_interp_time_regular()
    _rg = gen_bench_interp_time_regular()
    compare(_rb, _rg, "bench_interp_time_regular")
except Exception as _e:
    print(f"❌ L2 equivalence bench_interp_time_regular: setup error — {type(_e).__name__}: {_e}")

# L3 — multiple stations should keep only the first station after dropna.
try:
    edge_pd, edge_pl = _regular_pair(pd.DataFrame({
        "date": ["2020-01-01", "2020-01-02", "2020-01-03", "2020-01-04"],
        "value": [1.0, 2.0, 3.0, 4.0],
        "station_id": ["s2", "s1", "s2", "s1"],
    }))
    _rb = before_bench_interp_time_regular(edge_pd)
    _rg = gen_bench_interp_time_regular(edge_pl)
    assert _rg.get_column("station_id").unique().to_list() == ["s2"]
    compare(_rb, _rg, "L3 bench_interp_time_regular multiple stations")
except Exception as _e:
    print(f"❌ L3 bench_interp_time_regular multiple stations: {type(_e).__name__}: {_e}")

# L3 — rows dropped by null filtering can change which station is first.
try:
    edge_pd, edge_pl = _regular_pair(pd.DataFrame({
        "date": ["2020-01-01", "2020-01-02", "2020-01-03"],
        "value": [np.nan, 2.0, 3.0],
        "station_id": ["s1", "s2", "s2"],
    }))
    _rb = before_bench_interp_time_regular(edge_pd)
    _rg = gen_bench_interp_time_regular(edge_pl)
    assert _rg.get_column("station_id").unique().to_list() == ["s2"]
    compare(_rb, _rg, "L3 bench_interp_time_regular null first station")
except Exception as _e:
    print(f"❌ L3 bench_interp_time_regular null first station: {type(_e).__name__}: {_e}")

# L3 — all rows dropped should raise on both sides when first station is selected.
try:
    edge_pd, edge_pl = _regular_pair(pd.DataFrame({
        "date": [None, None],
        "value": [np.nan, np.nan],
        "station_id": ["s1", "s2"],
    }))
    before_err = gen_err = None
    try:
        before_bench_interp_time_regular(edge_pd)
    except Exception as e:
        before_err = type(e)
    try:
        gen_bench_interp_time_regular(edge_pl)
    except Exception as e:
        gen_err = type(e)
    assert before_err is not None and gen_err is not None
    print("✅ L3 bench_interp_time_regular all null: both sides raise")
except Exception as _e:
    print(f"❌ L3 bench_interp_time_regular all null: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_bench_interp_time_regular: OK, type= DataFrame
✅ L1 smoke before_bench_interp_time_regular: OK
✅ L2 equivalence bench_interp_time_regular: MATCH
✅ L3 edge bench_interp_time_regular multiple stations: MATCH
✅ L3 edge bench_interp_time_regular null first station: MATCH
✅ L3 bench_interp_time_regular all null: both sides raise
